# Reentrenar el extractor de comprobantes con el export de Document AI

1. Parámetros, Drive, código. 2. Export (desde Drive si ya se guardó, si no desde el bucket). 3. Conversión.
4. Modelo actual como referencia. 5. Entrenamiento. 6. Comparación en val, test una vez, prueba con una imagen.

**Dos modos, elegidos solos en el bloque 5:**
- **Split conservado**: si están los `manifest.json` del dataset anterior en `MyDrive/doctr_data/previous/doctr/{train,val,test}/`, los documentos anteriores quedan en su split, se continúa desde `r3_photo` (1 etapa) y la comparación con `r3_photo` es limpia.
- **Split nuevo**: si no están, se reparten los 988 documentos de nuevo y se entrena desde el preentrenado en 2 etapas. `r3_photo` vio en entrenamiento parte del test nuevo: su número sale inflado y solo sirve como referencia aproximada.

In [ ]:
# Bloque 1 · Parámetros
GCP_PROJECT = "noted-computing-469120-j9"  # proyecto de Google Cloud que tiene el bucket
GCS_EXPORT = ["gs://comprobantes-pago/train", "gs://comprobantes-pago/test"]  # carpetas del export de Document AI
# Document AI tiene la clase "fecha" escrita "facha", y "numero_cuenta" solo tiene 2 cajas
CONVERT_EXTRA = "--rename facha=fecha --drop numero_cuenta"
DRIVE = "/content/drive/MyDrive/doctr_data"
EXPORT_ZIP = f"{DRIVE}/export_docai.zip"  # copia del export en Drive (se crea en el bloque 4)
PREVIOUS = f"{DRIVE}/previous/doctr"  # manifests del dataset anterior, si los tienes
DATASET = "doctr_v2"  # dataset nuevo: se guarda como {DRIVE}/{DATASET}.zip
TEST_RATIO, VAL_RATIO, SEED = 0.20, 0.12, 42
BASELINE = "lw_detr_s_fields_r3_photo"  # modelo actual en {DRIVE}/checkpoints
RUN = "lw_detr_s_fields_r4"  # modelo nuevo (A)
RUN_B = f"{RUN}_ema_map"  # variante opcional (B)
BRANCH = "finetune-invoice"

In [ ]:
# Bloque 2 · Drive y GPU
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Bloque 3 · Código e instalación
REPO = "https://github.com/fercho1999leon/doctr.git"
!test -d /content/doctr || git clone -q --branch $BRANCH $REPO /content/doctr
%cd /content/doctr
!git pull -q origin $BRANCH && git log --oneline -1
!pip install -q -e . tqdm matplotlib rapidfuzz
import torch

print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Bloque 4 · Export de Document AI: desde Drive si ya está guardado, si no desde el bucket (y lo guarda en Drive)
import os

EXPORT_DIR = "/content/export"
sh = get_ipython().system
if os.path.exists(EXPORT_ZIP):
    print(f"Usando {EXPORT_ZIP}")
    sh(f'unzip -q -o "{EXPORT_ZIP}" -d /content')
else:
    from google.colab import auth

    auth.authenticate_user()
    sh(f"gcloud config set project {GCP_PROJECT}")
    for src in GCS_EXPORT:
        dst = f"{EXPORT_DIR}/{src.rstrip('/').split('/')[-1]}"  # /content/export/train, /content/export/test
        os.makedirs(dst, exist_ok=True)
        sh(f'gcloud storage rsync -r "{src}" "{dst}"')
    sh(f'cd /content && zip -qr "{EXPORT_ZIP}" export && echo "Guardado en {EXPORT_ZIP}"')
sh(f"echo JSON: $(find {EXPORT_DIR} -name '*.json' | wc -l)")

In [ ]:
# Bloque 5 · Convertir (conserva el split anterior si sus manifests están en Drive)
DATA = f"/content/datasets/{DATASET}"
ZIP = f"{DRIVE}/{DATASET}.zip"
LOG = f"{DATA}_convert.log"
HAS_PREVIOUS = all(os.path.exists(f"{PREVIOUS}/{s}/manifest.json") for s in ("train", "val", "test"))
print("MODO:", "split conservado (continúa desde r3_photo)" if HAS_PREVIOUS else "split nuevo (entrena desde el preentrenado)")
os.makedirs("/content/datasets", exist_ok=True)
if os.path.exists(ZIP):
    print(f"Reutilizando {ZIP}")
    sh(f'unzip -q -o "{ZIP}" -d /content/datasets')
else:
    keep = f"--keep-split {PREVIOUS}" if HAS_PREVIOUS else ""
    sh(
        f"python references/detection/convert_documentai.py --input {EXPORT_DIR} --output {DATA} {CONVERT_EXTRA} "
        f"{keep} --test-ratio {TEST_RATIO} --val-ratio {VAL_RATIO} --seed {SEED} --group-by-key "
        f"> {LOG} 2>&1; tail -12 {LOG}"
    )
    # Documentos omitidos (sin imagen embebida, sin páginas): deben ser 0
    sh(f"echo omitidos: $(grep -c '^.skip.' {LOG}); grep '^.skip.' {LOG} | head -5")
    sh(f"grep -E '^Kept from' {LOG}; cp {LOG} '{DRIVE}/'")
    sh(f'cd /content/datasets && zip -qr "{ZIP}" {DATASET} && echo "Guardado en {ZIP}"')

In [ ]:
# Bloque 6 · Revisión del dataset
import json

EXPECTED = {"cuenta_destino", "fecha", "nombre_cuenta_origen", "numero_comprobante", "numero_control", "valor_transferido"}
with open(f"{DATA}/audit.json", encoding="utf-8") as f:
    audit = json.load(f)
classes = set(audit["class_names"])
print("clases:", sorted(classes))
print("faltan:", sorted(EXPECTED - classes) or "-", "| sobran:", sorted(classes - EXPECTED) or "-")
for split, s in audit["splits"].items():
    print(f"\n{split}: {s['n_docs']} documentos")
    for c in sorted(classes):
        print(f"  {c:24s} {s['docs_with_class'][c]:5d} docs  ({s['presence_ratio'][c]:.0%})")
print("\nidentificadores compartidos entre splits:", len(audit["identifiers_shared_across_splits"]))


def split_ids(root, split):
    with open(f"{root}/{split}/manifest.json", encoding="utf-8") as f:
        return {d["id"] for d in json.load(f)["documents"]}


NEW_TEST = split_ids(DATA, "test") - (split_ids(PREVIOUS, "test") if HAS_PREVIOUS else set())
print(f"test: {len(split_ids(DATA, 'test'))} documentos, {len(NEW_TEST)} nuevos")

In [ ]:
# Bloque 7 · Modelo actual sobre el val nuevo (referencia)
CKPT = f"{DRIVE}/checkpoints"
EVAL = f"{DRIVE}/eval/{DATASET}"
os.makedirs(EVAL, exist_ok=True)
POST = "--top-k-per-class 1 --merge-fragments --fallback --straighten"


def evaluate(name, split="val"):
    sh(
        f"python references/detection/evaluate_fields.py --checkpoint {CKPT}/{name}.pt --data {DATA}/{split} "
        f"{POST} --output {EVAL}/{name}_{split} | tail -25"
    )


def report(name, split="val"):
    with open(f"{EVAL}/{name}_{split}.json", encoding="utf-8") as f:
        return json.load(f)


if not HAS_PREVIOUS:
    print("Aviso: split nuevo; r3_photo vio parte de estos documentos al entrenar, su resultado sale inflado.")
evaluate(BASELINE, "val")

In [ ]:
# Bloque 8 · Receta de entrenamiento (la de la última etapa de r3_photo) y comprobación contra su .json
import importlib.util
import shlex
import sys

BASE = (
    "--labels-name labels_layout.json --device 0 --amp --amp-dtype bfloat16 -b 8 --input_size 768 "
    "--epochs 40 --lr 2e-4 --backbone-lr 2e-4 --no-hflip --perspective 0.2 --photo-aug "
    "--early-stop --early-stop-epochs 12 --early-stop-delta 0.001 --save-interval-epoch -j 4"
)
# Solo en modo split nuevo: primera etapa desde el preentrenado (lr alto, sin aumentos de foto)
STAGE1 = (
    "--pretrained --labels-name labels_layout.json --device 0 --amp --amp-dtype bfloat16 -b 8 --input_size 768 "
    "--epochs 40 --lr 1e-3 --backbone-lr 1e-3 --no-hflip "
    "--early-stop --early-stop-epochs 10 --early-stop-delta 0.001 --save-interval-epoch -j 4"
)


def parsed_options(options):
    """The options train.py would run with, parsed by train.py itself."""
    spec = importlib.util.spec_from_file_location("layout_train", "references/layout/train.py")
    module = importlib.util.module_from_spec(spec)
    sys.path.insert(0, "references/layout")
    try:
        spec.loader.exec_module(module)
        argv = ["train.py", "lw_detr_s", *shlex.split(options), "--train_path", "-", "--val_path", "-"]
        saved, sys.argv = sys.argv, argv
        try:
            return vars(module.parse_args())
        finally:
            sys.argv = saved
    finally:
        sys.path.remove("references/layout")


IGNORED = {"train_path", "val_path", "output_dir", "name", "device", "workers", "resume", "wb", "clearml"}
with open(f"{CKPT}/{BASELINE}.json", encoding="utf-8") as f:
    base_args = json.load(f)["args"]
ours = parsed_options(BASE)
diff = {k: (base_args[k], v) for k, v in ours.items() if k in base_args and k not in IGNORED and base_args[k] != v}
print("BASE coincide con la receta de r3_photo" if not diff else "Distinto de r3_photo (antes -> ahora):")
for k, (old, new) in diff.items():
    print(f"  {k}: {old!r} -> {new!r}")

In [ ]:
# Bloque 9 · Entrenar A (el bloque largo)
def train(name, options, extra=""):
    if os.path.exists(f"{CKPT}/{name}.pt"):
        print(f"{name}.pt ya existe: se reutiliza (bórralo para reentrenar)")
        return
    sh(
        f"python references/layout/train.py lw_detr_s {options} {extra} --train_path {DATA}/train "
        f"--val_path {DATA}/val --output_dir {CKPT} --name {name}"
    )


if HAS_PREVIOUS:
    START = BASELINE  # se continúa desde r3_photo
else:
    START = f"{RUN}_s1"  # primera etapa desde el preentrenado
    train(START, STAGE1)
train(RUN, BASE, f"--resume {CKPT}/{START}.pt")
evaluate(RUN, "val")

In [ ]:
# Bloque 10 · (Opcional) Entrenar B: igual que A con EMA y selección por mAP
train(RUN_B, BASE, f"--resume {CKPT}/{START}.pt --ema --select-by map")
evaluate(RUN_B, "val")

In [ ]:
# Bloque 11 · Comparar en val
import pandas as pd
from scipy.stats import binomtest


def compare(names, split="val", base=BASELINE, ids=None):
    """Accuracy of the one value per field, over every document of the split or only `ids`."""

    def docs(name):
        return {d["id"]: d for d in report(name, split)["per_document"] if ids is None or d["id"] in ids}

    ref = docs(base)
    rows = []
    for name in [base, *names]:
        if not os.path.exists(f"{EVAL}/{name}_{split}.json"):
            print(f"{name}: sin reporte todavía")
            continue
        run = docs(name)
        row = {"modelo": name, "todos los requeridos": f"{sum(d['all_required_value_ok'] for d in run.values())}/{len(run)}"}
        for c in next(iter(run.values()))["fields"]:
            row[c] = f"{sum(d['fields'][c]['value_ok'] for d in run.values()) / len(run):.1%}"
        if name != base:
            only_base = sum(ref[k]["all_required_value_ok"] and not run[k]["all_required_value_ok"] for k in ref)
            only_run = sum(run[k]["all_required_value_ok"] and not ref[k]["all_required_value_ok"] for k in ref)
            n = only_base + only_run
            p = binomtest(only_run, n, 0.5).pvalue if n else 1.0
            row["vs base"] = f"+{only_run} / -{only_base} (p={p:.2f})"
        rows.append(row)
    return pd.DataFrame(rows).set_index("modelo")


compare([RUN, RUN_B])

In [ ]:
# Bloque 12 · Test, una sola vez
FINAL = RUN  # o RUN_B si ganó en val
evaluate(BASELINE, "test")
evaluate(FINAL, "test")
display(compare([FINAL], split="test"))
if HAS_PREVIOUS:
    display(compare([FINAL], split="test", ids=NEW_TEST))  # solo los documentos de test nuevos

In [ ]:
# Bloque 13 · Probar con una imagen
img = sorted(os.listdir(f"{DATA}/test/images"))[0]
sh(f'python references/detection/kie_inference.py --checkpoint {CKPT}/{FINAL}.pt {POST} --format receipt --json /content/prueba.json "{DATA}/test/images/{img}" --quiet')
sh("head -c 1500 /content/prueba.json")

Para desplegar el modelo nuevo en el servicio OCR del ERP (proveedor `doctr_fields`), copia **los dos archivos** de `MyDrive/doctr_data/checkpoints/`: `<FINAL>.pt` y `<FINAL>.json`. Sin el `.json` el modelo no carga.

Al terminar: `Entorno de ejecución > Desconectar y borrar el entorno de ejecución`.